<a href="https://colab.research.google.com/github/mathsacheantidote-spec/100-Days-of-python/blob/main/7Projects.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hotel & Restaurant Management System — Folder Structure

```
hotel_management/
│
├── main.py                  # Entry point, menu-driven interface
│
├── models/
│   ├── __init__.py
│   ├── room.py              # Abstract Room, DeluxeRoom, StandardRoom
│   ├── customer.py          # Customer class
│   ├── employee.py          # Employee class
│   ├── food.py              # FoodItem, Order classes
│   └── bill.py              # Bill class
│
├── core/
│   ├── __init__.py
│   ├── hotel.py             # Hotel (composition root)
│   └── reports.py           # Report generation (CSV)
│
├── utils/
│   ├── __init__.py
│   ├── exceptions.py        # Custom exceptions
│   ├── decorators.py        # Logging, validation decorators
│   └── helpers.py           # Static utilities (GST calc, date formatting)
│
└── data/                    # Auto-created at runtime
    ├── customers.json
    ├── rooms.json
    ├── employees.json
    ├── orders.json
    ├── bills.json
    ├── sales_report.csv
    ├── occupancy_report.csv
    └── employee_report.csv
```

## Module Responsibilities

| File | Responsibility |
|---|---|
| `models/room.py` | Abstract base + concrete room types |
| `models/customer.py` | Customer registration & booking history |
| `models/employee.py` | Employee records & roles |
| `models/food.py` | Menu items and order processing |
| `models/bill.py` | Invoice generation and tax |
| `core/hotel.py` | Composition root — owns all collections |
| `core/reports.py` | CSV report writers |
| `utils/exceptions.py` | All custom exception classes |
| `utils/decorators.py` | Logging, auth, and validation decorators |
| `utils/helpers.py` | Lambda sorts, GST, datetime helpers |
| `main.py` | Menu loop, load/save orchestration |

In [ ]:
class HotelBaseError(Exception):
    def __init__(self, message, code=0):
        super().__init__(message)
        self.message = message
        self.code = code
    def __str__(self):
        if self.code:
            return f"[Error {self.code}] {self.message}"
        return self.message
class RoomError(HotelBaseError):
    pass
class InvalidRoomNumberError(RoomError):
    def __init__(self, room_number):
        super().__init__(f"Room number '{room_number}' is not valid.", 101)
class RoomNotAvailableError(RoomError):
    def __init__(self, room_number):
        super().__init__(f"Room {room_number} is not available.", 102)
class RoomNotFoundError(RoomError):
    def __init__(self, identifier):
        super().__init__(f"No room found for '{identifier}'.", 103)
class RoomAlreadyExistsError(RoomError):
    def __init__(self, room_number):
        super().__init__(f"Room {room_number} already exists.", 104)
class CustomerError(HotelBaseError):
    pass
class InvalidCustomerIDError(CustomerError):
    def __init__(self, customer_id):
        super().__init__(f"Customer ID '{customer_id}' is invalid.", 201)
class CustomerAlreadyExistsError(CustomerError):
    def __init__(self, customer_id):
        super().__init__(f"Customer ID '{customer_id}' is registered.", 202)
class CustomerNotFoundError(CustomerError):
    def __init__(self, identifier):
        super().__init__(f"No customer found for '{identifier}'.", 203)
class OrderError(HotelBaseError):
    pass
class InvalidOrderIDError(OrderError):
    def __init__(self, order_id):
        super().__init__(f"Order ID '{order_id}' is invalid.", 301)
class OrderAlreadyCancelledError(OrderError):
    def __init__(self, order_id):
        super().__init__(f"Order '{order_id}' is already cancelled.", 302)
class EmptyOrderError(OrderError):
    def __init__(self):
        super().__init__("Cannot place an empty order.", 303)
class BillingError(HotelBaseError):
    pass
class InvalidBillAmountError(BillingError):
    def __init__(self, amount):
        super().__init__(f"Bill amount '{amount}' is invalid.", 401)
class BillNotFoundError(BillingError):
    def __init__(self, bill_id):
        super().__init__(f"No bill found for ID '{bill_id}'.", 402)
class FileError(HotelBaseError):
    pass
class DataFileNotFoundError(FileError):
    def __init__(self, filepath):
        super().__init__(f"Data file '{filepath}' was not found.", 501)
class DataCorruptedError(FileError):
    def __init__(self, filepath, reason=""):
        msg = f"File '{filepath}' is corrupted."
        if reason:
            msg += f" Reason: {reason}"
        super().__init__(msg, 502)
class InvalidInputError(HotelBaseError):
    def __init__(self, field, value=None):
        msg = f"Invalid input for field '{field}'"
        if value is not None:
            msg += f": '{value}'"
        super().__init__(msg + ".", 601)
errors = [
    InvalidInputError("Phone number", "9999"),
    RoomNotAvailableError(305),
    DataFileNotFoundError("data/rooms.json"),
]
for error in errors:
    print(error)


[Error 601] Invalid input for field 'Phone number': '9999'.
[Error 102] Room 305 is not available.
[Error 501] Data file 'data/rooms.json' was not found.


In [ ]:

from __future__ import annotations
import functools
import logging
import time
from datetime import datetime
from typing import Any, Callable, Optional, Tuple, Type
import os
logger = logging.getLogger("hotel_management")
def _configure_default_logging(
    log_file: str = "data/hotel_activity.log",
    level: int = logging.INFO,
) -> None:
    log_dir = os.path.dirname(log_file)
    if log_dir and not os.path.exists(log_dir):
        os.makedirs(log_dir, exist_ok=True)

    fmt = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"
    logging.basicConfig(
        level=level,
        format=fmt,
        handlers=[
            logging.FileHandler(log_file, encoding="utf-8"),
            logging.StreamHandler(),
        ],
    )
F = Callable[..., Any]
def log_action(action_name: Optional[str] = None) -> Callable[[F], F]:
    def decorator(func: F) -> F:
        label = action_name or func.__name__
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            start = time.perf_counter()
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            logger.info("► [%s] STARTED at %s | args=%r kwargs=%r",
                        label, timestamp, args, kwargs)
            try:
                result = func(*args, **kwargs)
                elapsed = (time.perf_counter() - start) * 1000
                logger.info("✔ [%s] SUCCESS — %.1f ms", label, elapsed)
                return result
            except Exception as exc:
                elapsed = (time.perf_counter() - start) * 1000
                logger.error(
                    "✘ [%s] FAILED after %.1f ms — %s: %s",
                    label, elapsed, type(exc).__name__, exc,
                )
                raise
        return wrapper
    return decorator
def log_booking(func: F) -> F:
    @functools.wraps(func)
    @log_action("Room Booking")
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        room_number = kwargs.get("room_number") or (args[1] if len(args) > 1 else "N/A")
        customer_id = kwargs.get("customer_id") or (args[2] if len(args) > 2 else "N/A")

        logger.info(
            "  ↳ Booking attempt — room=%s, customer=%s",
            room_number, customer_id,
        )
        return func(*args, **kwargs)
    return wrapper
def log_food_order(func: F) -> F:
    @functools.wraps(func)
    @log_action("Food Ordering")
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        customer_id = kwargs.get("customer_id") or (args[1] if len(args) > 1 else "N/A")
        items = kwargs.get("items") or (args[2] if len(args) > 2 else [])
        item_count = len(items) if isinstance(items, (list, tuple)) else "?"
        logger.info(
            "  ↳ Order attempt — customer=%s, item_count=%s",
            customer_id, item_count,
        )
        return func(*args, **kwargs)
    return decorator
def log_bill(func: F) -> F:
    @functools.wraps(func)
    @log_action("Bill Generation")
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        customer_id = kwargs.get("customer_id") or (args[1] if len(args) > 1 else "N/A")

        logger.info("  ↳ Bill generation — customer=%s", customer_id)
        result = func(*args, **kwargs)
        if hasattr(result, "total_amount"):
            logger.info(
                "  ↳ Invoice total: ₹%.2f (customer=%s)",
                result.total_amount, customer_id,)
        return result
    return decorator
def validate_input(*required_kwargs: str) -> Callable[[F], F]:
    def decorator(func: F) -> F:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            for field in required_kwargs:
                value = kwargs.get(field)
                if value is None or value == "":
                    raise ValueError(
                        f"Required argument '{field}' is missing or empty "
                        f"in call to '{func.__name__}'."
                    )
            return func(*args, **kwargs)
        return wrapper
    return decorator
def retry(
    max_attempts: int = 3,
    delay_seconds: float = 1.0,
    exceptions: Tuple[Type[Exception], ...] = (Exception,),
) -> Callable[[F], F]:
    def decorator(func: F) -> F:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            last_exc: Optional[Exception] = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:   # type: ignore[misc]
                    last_exc = exc
                    logger.warning(
                        "Retry %d/%d for '%s' — %s: %s",
                        attempt, max_attempts, func.__name__,
                        type(exc).__name__, exc,
                    )
                    if attempt < max_attempts:
                        time.sleep(delay_seconds)
            logger.error(
                "All %d attempts failed for '%s'.", max_attempts, func.__name__
            )
            raise last_exc
        return wrapper
    return decorator
_configure_default_logging()
@log_action("My Test Function")
def my_test_function(param1: str, param2: int):
    logger.info(f"Inside my_test_function with {param1} and {param2}")
    return param1 * param2
@log_action()
def another_function():
    logger.warning("This is a warning from another_function.")
    raise ValueError("Something went wrong!")
print("-- Executing logged functions ---")
my_test_function("test", 3)
try:
    another_function()
except ValueError as e:
    logger.error(f"Caught expected error: {e}")
print("-- Log entries should now be visible in output and data/hotel_activity.log ---")


ERROR:hotel_management:✘ [another_function] FAILED after 1.0 ms — ValueError: Something went wrong!
ERROR:hotel_management:Caught expected error: Something went wrong!


-- Executing logged functions ---
-- Log entries should now be visible in output and data/hotel_activity.log ---


In [ ]:
from __future__ import annotations
import functools
import logging
import time
from datetime import datetime
from typing import Any, Callable, Optional, Tuple, Type
import os
logger = logging.getLogger("hotel_management")
def _configure_default_logging(
    log_file: str = "data/hotel_activity.log",
    level: int = logging.INFO,
) -> None:
    log_dir = os.path.dirname(log_file)
    if log_dir and not os.path.exists(log_dir):
        os.makedirs(log_dir, exist_ok=True)

    fmt = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"
    logging.basicConfig(
        level=level,
        format=fmt,
        handlers=[
            logging.FileHandler(log_file, encoding="utf-8"),
            logging.StreamHandler(),
        ],
    )
F = Callable[..., Any]
def log_action(action_name: Optional[str] = None) -> Callable[[F], F]:
    def decorator(func: F) -> F:
        label = action_name or func.__name__
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            start = time.perf_counter()
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            logger.info("► [%s] STARTED at %s | args=%r kwargs=%r",
                        label, timestamp, args, kwargs)
            try:
                result = func(*args, **kwargs)
                elapsed = (time.perf_counter() - start) * 1000
                logger.info("✔ [%s] SUCCESS — %.1f ms", label, elapsed)
                return result
            except Exception as exc:
                elapsed = (time.perf_counter() - start) * 1000
                logger.error(
                    "✘ [%s] FAILED after %.1f ms — %s: %s",
                    label, elapsed, type(exc).__name__, exc,
                )
                raise
        return wrapper
    return decorator
def log_booking(func: F) -> F:
    @functools.wraps(func)
    @log_action("Room Booking")
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        room_number = kwargs.get("room_number") or (args[1] if len(args) > 1 else "N/A")
        customer_id = kwargs.get("customer_id") or (args[2] if len(args) > 2 else "N/A")

        logger.info(
            "  ↳ Booking attempt — room=%s, customer=%s",
            room_number, customer_id,
        )
        return func(*args, **kwargs)
    return wrapper
def log_food_order(func: F) -> F:
    @functools.wraps(func)
    @log_action("Food Ordering")
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        customer_id = kwargs.get("customer_id") or (args[1] if len(args) > 1 else "N/A")
        items = kwargs.get("items") or (args[2] if len(args) > 2 else [])
        item_count = len(items) if isinstance(items, (list, tuple)) else "?"
        logger.info(
            "  ↳ Order attempt — customer=%s, item_count=%s",
            customer_id, item_count,
        )
        return func(*args, **kwargs)
    return decorator
def log_bill(func: F) -> F:
    @functools.wraps(func)
    @log_action("Bill Generation")
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        customer_id = kwargs.get("customer_id") or (args[1] if len(args) > 1 else "N/A")

        logger.info("  ↳ Bill generation — customer=%s", customer_id)
        result = func(*args, **kwargs)
        if hasattr(result, "total_amount"):
            logger.info(
                "  ↳ Invoice total: ₹%.2f (customer=%s)",
                result.total_amount, customer_id,)
        return result
    return decorator
def validate_input(*required_kwargs: str) -> Callable[[F], F]:
    def decorator(func: F) -> F:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            for field in required_kwargs:
                value = kwargs.get(field)
                if value is None or value == "":
                    raise ValueError(
                        f"Required argument '{field}' is missing or empty "
                        f"in call to '{func.__name__}'."
                    )
            return func(*args, **kwargs)
        return wrapper
    return decorator
def retry(
    max_attempts: int = 3,
    delay_seconds: float = 1.0,
    exceptions: Tuple[Type[Exception], ...] = (Exception,),
) -> Callable[[F], F]:
    def decorator(func: F) -> F:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            last_exc: Optional[Exception] = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:   # type: ignore[misc]
                    last_exc = exc
                    logger.warning(
                        "Retry %d/%d for '%s' — %s: %s",
                        attempt, max_attempts, func.__name__,
                        type(exc).__name__, exc,
                    )
                    if attempt < max_attempts:
                        time.sleep(delay_seconds)
            logger.error(
                "All %d attempts failed for '%s'.", max_attempts, func.__name__
            )
            raise last_exc
        return wrapper
    return decorator
_configure_default_logging()
@log_action("My Test Function")
def my_test_function(param1: str, param2: int):
    logger.info(f"Inside my_test_function with {param1} and {param2}")
    return param1 * param2
@log_action()
def another_function():
    logger.warning("This is a warning from another_function.")
    raise ValueError("Something went wrong!")
print("-- Executing logged functions ---")
my_test_function("test", 3)
try:
    another_function()
except ValueError as e:
    logger.error(f"Caught expected error: {e}")
print("-- Log entries should now be visible in output and data/hotel_activity.log ---")

ERROR:hotel_management:✘ [another_function] FAILED after 2.7 ms — ValueError: Something went wrong!
ERROR:hotel_management:Caught expected error: Something went wrong!


-- Executing logged functions ---
-- Log entries should now be visible in output and data/hotel_activity.log ---


In [ ]:
# This cell previously contained redundant and syntactically incorrect code for the Bill class.
# The full and correct Bill class definition is now located in cell d9PvCPNf_ADJ.

In [ ]:
import random
import datetime

# Global List Declaration
name = []
phno = []
add = []
checkin = []
checkout = []
room = []
price = []
rc = []
p = []
roomno = []
custid = []
day = []

# Global Variable Declaration
i = 0

# Home Function
def Home():
    while True:
        print("				 WELCOME TO HOTEL TAJ")
        print("			 1 Booking")
        print("			 2 Rooms Info")
        print("			 3 Room Service(Menu Card)")
        print("			 4 Payment")
        print("			 5 Record")
        print("			 0 Exit")

        try:
            ch = int(input("-> "))
        except ValueError:
            print("Invalid input. Please enter a number.")
            continue

        if ch == 1:
            Booking()
        elif ch == 2:
            Rooms_Info()
        elif ch == 3:
            restaurant()
        elif ch == 4:
            Payment()
        elif ch == 5:
            Record()
        elif ch == 0:
            print("Exiting Hotel . Thank you!")
            break
        else:
            print("Invalid Choice! Please try again.")

# Date validation function (modified to return boolean)
def date_validator(day_val, month_val, year_val):
    try:
        datetime.datetime(year_val, month_val, day_val)
        return True
    except ValueError:
        return False

# Booking Function
def Booking():
    global i
    print("\n---- BOOKING ROOMS ----\n")

    while True:
        n = str(input("Name: "))
        p1 = str(input("Phone No.: "))
        a = str(input("Address: "))
        if n != "" and p1 != "" and a != "":
            name.append(n)
            phno.append(p1)
            add.append(a)
            break
        else:
            print("Name, Phone no. & Address cannot be empty..!!")

    ci_day, ci_month, ci_year = 0, 0, 0 # Initialize to avoid potential UnboundLocalError if loop doesn't break
    # Check-In Date Input Loop
    while True:
        cii_str = input("Check-In (dd/mm/yyyy): ")
        cii_parts = cii_str.split('/')
        if len(cii_parts) == 3:
            try:
                ci_day, ci_month, ci_year = int(cii_parts[0]), int(cii_parts[1]), int(cii_parts[2])
                if date_validator(ci_day, ci_month, ci_year):
                    checkin.append(cii_str)
                    break
                else:
                    print("Invalid check-in date. Please enter a valid date.")
            except ValueError:
                print("Invalid check-in date. Please use numbers for day, month, and year.")
        else:
            print("Invalid check-in date format. Please use dd/mm/yyyy.")

    # Check-Out Date Input Loop
    while True:
        coo_str = input("Check-Out (dd/mm/yyyy): ")
        coo_parts = coo_str.split('/')
        if len(coo_parts) == 3:
            try:
                co_day, co_month, co_year = int(coo_parts[0]), int(coo_parts[1]), int(coo_parts[2])
                if date_validator(co_day, co_month, co_year):
                    # Check if Check-out is after Check-in
                    d1 = datetime.datetime(ci_year, ci_month, ci_day)
                    d2 = datetime.datetime(co_year, co_month, co_day)
                    if d2 <= d1:
                        print("\nCheck-Out date must fall after Check-In. Please try again.\n")
                    else:
                        checkout.append(coo_str)
                        d = (d2 - d1).days
                        day.append(d)
                        break
                else:
                    print("Invalid check-out date. Please enter a valid date.")
            except ValueError:
                print("Invalid check-out date. Please use numbers for day, month, and year.")
        else:
            print("Invalid check-out date format. Please use dd/mm/yyyy.")

    print("\n----SELECT ROOM TYPE----")
    print(" 1. Standard Non-AC - Rs. 3500/day")
    print(" 2. Standard AC - Rs. 4000/day")
    print(" 3. 3-Bed Non-AC - Rs. 4500/day")
    print(" 4. 3-Bed AC - Rs. 5000/day")

    while True:
        try:
            ch = int(input("-> "))
            if ch == 1:
                room.append('Standard Non-AC')
                price.append(3500)
                break
            elif ch == 2:
                room.append('Standard AC')
                price.append(4000)
                break
            elif ch == 3:
                room.append('3-Bed Non-AC')
                price.append(4500)
                break
            elif ch == 4:
                room.append('3-Bed AC')
                price.append(5000)
                break
            else:
                print("Wrong choice. Please select a valid option (1-4).")
        except ValueError:
            print("Invalid input. Please enter a number.")

    rn = random.randrange(40) + 300
    cid = random.randrange(40) + 10

    while rn in roomno or cid in custid:
        rn = random.randrange(60) + 300
        cid = random.randrange(60) + 10

    roomno.append(rn)
    custid.append(cid)
    rc.append(0)
    p.append(0)

    print("\n*** ROOM BOOKED SUCCESSFULLY ***")
    print(f"Room No. - {rn}")
    print(f"Customer Id - {cid}")
    i += 1 # Increment global counter

# Rooms Info Function
def Rooms_Info():
    print("\n------ HOTEL ROOMS INFO ------\n")
    print("STANDARD NON-AC - Double Bed, TV, Telephone, Balcony, Washroom\n")
    print("STANDARD AC - Double Bed, TV, Telephone, Balcony, Washroom, AC\n")
    print("3-Bed NON-AC - 3 Beds, TV, Telephone, Balcony, Washroom\n")
    print("3-Bed AC - 3 Beds, TV, Telephone, Balcony, Washroom, AC\n")

# Restaurant Function
def restaurant():
    try:
        ph = int(input("Customer Id: "))
    except ValueError:
        print("Invalid input. Please enter a number for Customer Id.")
        return

    global i
    f = 0
    r = 0
    for n in range(0, i):
        if custid[n] == ph and p[n] == 0:
            f = 1
            print("\n---- HOTEL TAJ MENU ----")
            print("39 Masala Dosa - 130")
            print("32 Butter Naan - 20")
            print("40 Paneer Dosa - 130")
            print("Press 0 to end order")

            while True:
                try:
                    ch = int(input("-> "))
                    if ch == 39:
                        r += 130
                    elif ch == 32:
                        r += 20
                    elif ch == 40:
                        r += 130
                    elif ch == 0:
                        break
                    else:
                        print("Invalid choice.")
                except ValueError:
                    print("Invalid input. Please enter a number.")

            print(f"\nTotal Restaurant Bill: Rs. {r}")
            rc[n] += r
            break
    if f == 0:
        print("Invalid Customer Id")

# Payment Function
def Payment():
    ph = str(input("Phone Number: "))
    global i
    f = 0

    for n in range(0, i):
        if ph == phno[n]:
            if p[n] == 0:
                f = 1
                print("\n---- PAYMENT ----")
                print("1- Credit/Debit Card")
                print("2- Paytm/PhonePe")
                print("3- Using UPI")
                print("4- Cash")
                while True:
                    try:
                        _ = int(input("-> ")) # Payment method choice
                        break
                    except ValueError:
                        print("Invalid input. Please enter a number.")

                total_amt = (price[n] * day[n]) + rc[n]
                print(f"\nAmount to Pay: Rs. {total_amt}")

                ch = input("Confirm Payment? (y/n): ")
                if ch.lower() == 'y':
                    print("\n---- HOTEL TAJ BILL ----")
                    print(f"Name: {name[n]}")
                    print(f"Phone No.: {phno[n]}")
                    print(f"Address: {add[n]}")
                    print(f"Check-In: {checkin[n]}")
                    print(f"Check-Out: {checkout[n]}")
                    print(f"Room Type: {room[n]}")
                    print(f"Room Charges: {price[n] * day[n]}")
                    print(f"Restaurant Charges: {rc[n]}")
                    print(f"Total Amount Paid: {total_amt}")
                    print("\nThank You. Visit Again!\n")

                    p[n] = 1
                    roomno[n] = 0
                    custid[n] = 0
            else:
                print("Payment already done.")
            f = 1
            break
    if f == 0:
        print("Invalid Customer Id")

# Record Function
def Record():
    if phno:
        print("\n---- HOTEL RECORD ----")
        print("| Name | Phone No. | Address | Check-In | Check-Out | Room Type | Price/day |")
        print("-" * 90)
        for n in range(0, i):
            print(f"| {name[n]} | {phno[n]} | {add[n]} | {checkin[n]} | {checkout[n]} | {room[n]} | {price[n]} |")
    else:
        print("No Records Found")

Home()

				 WELCOME TO HOTEL TAJ
			 1 Booking
			 2 Rooms Info
			 3 Room Service(Menu Card)
			 4 Payment
			 5 Record
			 0 Exit
-> 1

---- BOOKING ROOMS ----

Name: Sreyansh dixit
Phone No.: 7676767676
Address: KP2 greater noida
Check-In (dd/mm/yyyy): 12/06/2026
Check-Out (dd/mm/yyyy): 14/06/2026

----SELECT ROOM TYPE----
 1. Standard Non-AC - Rs. 3500/day
 2. Standard AC - Rs. 4000/day
 3. 3-Bed Non-AC - Rs. 4500/day
 4. 3-Bed AC - Rs. 5000/day
-> 3

*** ROOM BOOKED SUCCESSFULLY ***
Room No. - 335
Customer Id - 42
				 WELCOME TO HOTEL TAJ
			 1 Booking
			 2 Rooms Info
			 3 Room Service(Menu Card)
			 4 Payment
			 5 Record
			 0 Exit
-> 2

------ HOTEL ROOMS INFO ------

STANDARD NON-AC - Double Bed, TV, Telephone, Balcony, Washroom

STANDARD AC - Double Bed, TV, Telephone, Balcony, Washroom, AC

3-Bed NON-AC - 3 Beds, TV, Telephone, Balcony, Washroom

3-Bed AC - 3 Beds, TV, Telephone, Balcony, Washroom, AC

				 WELCOME TO HOTEL TAJ
			 1 Booking
			 2 Rooms Info
			 3 Room Service(M

Project 1: A Personal Finance Tracker That Reads Real Bank Data
Skills learned: CSV parsing, data cleaning, Pandas fundamentals, handling messy real-world data.

The moment you stop working with made-up data someone else cleaned for you and start working with data that came from the actual world — which is always messier than you expect — something important happens. You learn that data is a problem first and an asset second.

Download three months of your own bank statements as CSV files. Your bank almost certainly offers this, and almost every bank formats it differently.

Now try to build something that reads all three files, handles the inconsistencies, categorizes transactions, and produces a monthly summary. You’ll encounter date fields formatted four different ways, encoding problems, mysterious extra columns, and descriptions that are technically information but practically useless. You’ll write code to clean data, which will break on edge cases you didn’t anticipate, which you’ll then fix, which will break on other edge cases.

This is real data work. It’s what data analysts and data scientists actually spend most of their time doing — not modeling, not visualization, cleaning. The Stack Overflow question “how do I handle this encoding error” that you’ll hit approximately three times during this project represents real professional skill, not toy-project skill.

When you’re done, you’ll have something actually useful. And you’ll have touched Pandas in a way that a dozen tutorials couldn’t give you, because you’ll have used it on data you didn’t control.

To build a production-grade UPI Transaction & Limit Management Platform, we must transition away from standard tutorial configurations. Below is an exhaustive breakdown of the technical components, why they are selected, and an exact engineering timeline to build the core architecture.
------------------------------
## 🛠️ The Production FinTech Technology Stack
For a high-throughput, secure financial parsing app, choose Stack A (Python Ecosystem) for deep data manipulation or Stack B (TypeScript Ecosystem) for rapid, asynchronous execution.
## ⚙️ Option A: The Python Systems Stack (Recommended for Data Heavy Operations)

* Core API Gateway: FastAPI + Uvicorn. Chosen for its native asynchronous request handling, extremely low latency overhead, and automatic validation using Pydantic V2.
* Task Distribution Worker: Celery. Handles long-running report tasks and transaction matching processes away from the client HTTP loop.
* Database Object-Relational Mapping (ORM): SQLAlchemy (Async engine) + Alembic for schema version migrations.

## ⚡ Option B: The TypeScript Systems Stack (Recommended for Event-Driven Architectures)

* Core API Gateway: NestJS (Fastify adapter). Provides a strict module-based architecture with structural dependency injection, making it highly secure and testable.
* Task Distribution Worker: BullMQ. A heavy-duty, Redis-backed queueing mechanism built explicitly for low-overhead Node.js systems.
* Database Object-Relational Mapping (ORM): Prisma or TypeORM.

## 🗄️ Shared Infrastructure Matrix (Mandatory for both Stacks)

* Primary Data Store: PostgreSQL. Selected for strict ACID compliance, transactional guarantees (crucial when dealing with ledger adjustments), and its advanced row-locking mechanics (SELECT FOR UPDATE).
* Message Broker & Cache: Redis. Acts as the caching layer for fast user limit counters and serves as the state engine backend for your task workers.
* Telemetry & Observability: Prometheus (for metric instrumentation scraping) + Grafana (for visual telemetry analysis).
* Containerization: Docker and Docker Compose for local infrastructure reproduction.

------------------------------
## 🗺️ Granular Technical Implementation Roadmap

graph TD
    Week1_3[Weeks 1-3: Relational Schema & Dockerization] --> Week4_6[Weeks 4-6: Text Ingestion & Multi-Tenant Regex Engines]
    Week4_6[Weeks 4-6: Text Ingestion & Multi-Tenant Regex Engines] --> Week7_9[Weeks 7-9: Async Queue Workers & Row Isolation]
    Week7_9[Weeks 7-9: Async Queue Workers & Row Isolation] --> Week10_12[Weeks 10-12: PDF Automation & Telemetry Metrics]
    
    style Week1_3 fill:#f1f5f9,stroke:#64748b,stroke-width:2px
    style Week4_6 fill:#f1f5f9,stroke:#64748b,stroke-width:2px
    style Week7_9 fill:#f1f5f9,stroke:#64748b,stroke-width:2px
    style Week10_12 fill:#f1f5f9,stroke:#64748b,stroke-width:2px

## 🗓️ Phase 1: Storage Layer & Infrastructure (Weeks 1–3)
Objective: Architect an isolated relational database model capable of preventing race conditions during parallel transaction writes.

* Step 1.1: Initialize the Environment
Configure a multi-container local workspace using Docker Compose. Ensure your backend application layer, PostgreSQL container, and Redis broker run on a isolated internal virtual network interface.
* Step 1.2: Design the Relational Tables
Write database migration files (Alembic or Prisma) defining the tables below. Enforce index constraints on highly targeted query points:
* users: Primary user profiles (id [UUID], email, created_at).
   * limits: Track budget targets (id, user_id [FK], category, monthly_amount [Decimal(12,2)], current_spent [Decimal(12,2)]). Add a composite index on (user_id, category).
   * transactions: Ledger records (id, user_id [FK], raw_text, amount [Decimal(12,2)], merchant_name, category, timestamp).
* Step 1.3: Prevent Limit Overwriting (Race Conditions)
When two UPI transactions happen at almost the exact same second, your backend will read the current budget value simultaneously, leading to an incorrect total calculation. You must write an explicit row-level locking pattern to prevent this:

-- Standard safe execution loop: Lock the specific row until the write completesSELECT current_spent FROM limits WHERE user_id = :user_id AND category = :category FOR UPDATE;


------------------------------
## 🗓️ Phase 2: Ingestion & Parsing Engines (Weeks 4–6)
Objective: Build a regex matching matrix that breaks down variable incoming text notification structures into clean JSON attributes.

* Step 2.1: Map the Notification String Formats
Indian UPI notification texts vary heavily across banks and applications (HDFC, SBI, GPay, PhonePe). Compile an array of exact test structures to match against:
* Format A: "Debited for Rs.150.00 from a/c X1234 to VPA merchant@ybl on 06-09-26."
   * Format B: "Sent Rs.500 to Swiggy via UPI Ref 624590123."
* Step 2.2: Implement the Regex Compilers
Write isolated text extractors using strict named capture groups to pull parameters cleanly out of raw strings:

import re
# Named capture groups pull structural components regardless of placementUPI_DEBIT_PATTERN = r"(?:Debited|Sent)\s+(?:for\s+)?(?:Rs\.|INR)\s*(?P<amount>\d+\.\d{2})\s+.*?(?:to|via)\s+(?P<merchant>[\w@\.\s]+)"
def parse_incoming_sms(raw_text: str):
    match = re.search(UPI_DEBIT_PATTERN, raw_text, re.IGNORECASE)
    if match:
        return {
            "amount": float(match.group("amount")),
            "merchant": match.group("merchant").strip()
        }
    return None

* Step 2.3: Build the Categorization Matrix
Map extracted merchant strings to structured business categories using a substring token matrix (e.g., matching zomato, swiggy, blinkit directly to a 'Food & Dining' value string).

------------------------------
## 🗓️ Phase 3: Event Decoupling & Queue Resiliency (Weeks 7–9)
Objective: Protect your ingestion layer from dropping transactions under high-frequency load by introducing message brokers.

* Step 3.1: Offload Work to Task Queues
When an incoming transaction payload hits your server, do not run database writes or analytics calculations on the primary request thread. Instead, push the raw event payload immediately to your Redis Queue via Celery or BullMQ, and return an instant HTTP 202 Accepted status code back to the client device interface.
* Step 3.2: Implement the Asynchronous Execution Pipeline
Configure background worker servers whose entire job is to pull payloads off the queue one by one, execute regex extractions, run database transactions, and evaluate budget safety lines.
* Step 3.3: Write Automated Integration Tests
Build a comprehensive automated test file (test_pipeline.py) that initializes a mock database instance, loops through 50+ diverse, messy transaction text samples, and asserts that the pipeline parses the amounts, formats the dates, and flags budget overflows correctly.

------------------------------
## 🗓️ Phase 4: Analytical Reporting & Telemetry Monitoring (Weeks 10–12)
Objective: Autogenerate monthly data summary assets and construct live platform performance tracking dashboards.

* Step 4.1: Code the PDF Compilation Microservice
Write an automated script using libraries like ReportLab (Python) or PDFKit (Node.js). It should extract all transaction records for a given user from the past 30 days and build a clean, structured financial report detailing total outflow metrics, category spend charts, and threshold alert lists.
* Step 4.2: Instrument System Metrics (Prometheus)
Expose a clean /metrics path using an official Prometheus client library to count internal system behaviors in real time:
* upi_transactions_processed_total (Counter)
   * upi_budget_limit_breaches_total (Counter)
   * http_request_duration_seconds (Histogram tracking API speed latency metrics)
* Step 4.3: Deploy the Grafana Monitoring Board
Connect your running Grafana engine to the Prometheus data stream. Create a live telemetry monitoring wall displaying: API execution speed in milliseconds, Current queue length backlogs, and a live count showing Budget threshold alerts triggered across the system per hour.

------------------------------
## 🛡️ Core Enterprise Security & Compliance Guardrails
To prove to enterprise engineering interviewers that you possess deep fintech acumen, your system must actively enforce these three core production security standards:

   1. Multi-Tenant Query Isolation: Implement an automatic global query filter inside your ORM layer. Every database call must append an explicit checking constraint (WHERE user_id = current_authenticated_user_id) to ensure database reads can never accidentally leakage records between user scopes.
   2. Data Sanitization & Encryption: Mask sensitive text data elements (such as personal bank account numbers, UPI PIN confirmation fragments, or full names) using regex sanitization filters before storing transaction rows.
   3. Strict Token Serialization Validation: Wrap your incoming network request boundaries behind automated serialization layers (Pydantic V2 schemas or Zod objects). If a payload parameter passes corrupt symbols or a malformed data type, your application must drop the request instantly at the system border to mitigate SQL Injection or data corruptions.




In [ ]:
# 1. Install PostgreSQL server and its client binaries
!apt-get -y update
!apt-get -y install postgresql postgresql-contrib

# 2. Start the database service in the Linux background
!sevice postgresql start
# 3. Create our database user, password, and the database instance using SQL commands
!sudo -u postgresql -c "CREATE USER fintech_admin WITH PASSWORD 'secure_api_vault_api_2026';"
!sudo -u postgresql -c "CREATE DATABASE upi_tracker_ledger OWNER fintech_admin;"
!sudo -u postgresql -c "GRANT ALL PRIVILEDGES ON DATABASE upi_tracker_ledger TO  fintech_admin;"

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [4,685 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [113 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,316 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,287 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,706 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packa

In [ ]:
# asyncpg is the fastest async database driver for PostgreSQL in the Python ecosystem
!pip install sqlalchemy asyncpg pydantic pydantic-settings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.8 MB/s eta 0:00:00


In [ ]:
!service postgresql status

In [ ]:
# 1. Force start the server by pointing the logfile inside the postgres-owned directory
!sudo -u postgres /usr/lib/postgresql/14/bin/pg_ctl -D /var/lib/postgresql/14/main -l /var/lib/postgresql/14/main/server.log start

# 2. Give the database engine 2 seconds to finish opening sockets safely
!sleep 2

# 5. Provision the necessary ledger credentials
!sudo -u postgresql -c "CREATE USER fintech_admin WITH P ASSWORD 'secure_api_vault_api_2026';"
!sudo -u postgresql -c "CREATE DATABASE upi_tracker_ledger OWNER fintech_admin;"
!sudo -u postgresql -c "GRANT ALL PRIVILEDGES ON DATABASE upi_tracker_ledger TO  fintech_admin;"



waiting for server to start.... done
server started
usage: sudo -h | -K | -k | -V
usage: sudo -v [-ABknS] [-g group] [-h host] [-p prompt] [-u user]
usage: sudo -l [-ABknS] [-g group] [-h host] [-p prompt] [-U user] [-u user]
            [command]
usage: sudo [-ABbEHknPS] [-r role] [-t type] [-C num] [-D directory] [-g group]
            [-h host] [-p prompt] [-R directory] [-T timeout] [-u user]
            [VAR=value] [-i|-s] [<command>]
usage: sudo -e [-ABknS] [-r role] [-t type] [-C num] [-D directory] [-g group]
            [-h host] [-p prompt] [-R directory] [-T timeout] [-u user] file ...
usage: sudo -h | -K | -k | -V
usage: sudo -v [-ABknS] [-g group] [-h host] [-p prompt] [-u user]
usage: sudo -l [-ABknS] [-g group] [-h host] [-p prompt] [-U user] [-u user]
            [command]
usage: sudo [-ABbEHknPS] [-r role] [-t type] [-C num] [-D directory] [-g group]
            [-h host] [-p prompt] [-R directory] [-T timeout] [-u user]
            [VAR=value] [-i|-s] [<command>]
usag

In [ ]:
# 4. Check the operational status of the service cluster
!service postgresql status


In [ ]:
import psycopg2
# Connect via native driver to apply our baseline structural constraints
conn = psycopg2.connect(
    dbname = "upi_tracker",
    user = "fintech_admin",
    password =  "security_api_vault_pass_2026",
    host = "localhost",
    port = "5432"
)
cursor = conn.cursor()

# Execute Data Definition Language (DDL)
cursor.execute("""
CREATE EXTENSIONS IF NOT EXISTS "uuid-ossp";

CREATE TABLE IF NOT EXISTS users
(id UUID PRIMARY DEFAULT KEY gen_random_uuid(),
email VARCHAR(225) UNIQUE NOT NULL,
created_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS limits(
id UUID PRIMARY DEFAULT KEY gen_random_uuid(),
uuid USER_ID NOT NULL,
category VARCHAR(50) NOT NULL,
monthly_amount NUMERIC(12, 2) NOT NULL,
current_stamp NUMERIC(12, 2) DEFAULT 0.00,
updated_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP,
CONSTRAINT fk_user FOREIGN KEY(user_id) REFERENCES(user_id) ON DELETE CASCADE,
CONSTRAINT unique_user_category UNIQUE( user_id, category)
);

CREATE INDEX IF NOT EXISTS idx_limits_user_catergory on LIMITS(user_id, category);
""")
conn.commit()
cursor.close()
conn.close()
print("Database Schema initialized with optimized indices successfully.")

OperationalError: connection to server at "localhost" (::1), port 5432 failed: FATAL:  role "fintech_admin" does not exist


In [ ]:
import asyncio
from sqlalchemy.ext.asyncio import create_async_engine, AsyncSession, async_sessionmaker
from sqlalchemy.orm import declarative_base
# The 'postgresql+asyncpg' scheme tells SQLAlchemy to run 100% asynchronously
